In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Carregar dados já limpos da etapa anterior
df = pd.read_csv("../data/processed/nvda_clean.csv", index_col=0, parse_dates=True)

df.head()

,close
Date,
2018-01-02,4.928532
2018-01-03,5.252899
2018-01-04,5.280590
2018-01-05,5.325338
2018-01-08,5.488509


In [2]:
LOOKBACK = 30   # janela de 30 dias
HORIZON = 1     # prever 1 dia à frente

print("Lookback:", LOOKBACK)
print("Horizonte:", HORIZON)

Lookback: 30
Horizonte: 1


In [3]:
def create_windows(series, lookback, horizon=1):
    """
    Transforma uma série temporal em janelas supervisionadas.
    
    Retorna:
    X -> shape (n_samples, lookback)
    y -> shape (n_samples,)
    """
    X, y = [], []

    values = series.values

    for i in range(len(values) - lookback - horizon + 1):
        X.append(values[i : i + lookback])
        y.append(values[i + lookback + horizon - 1])

    return np.array(X), np.array(y)

In [4]:
# Separar novamente os splits (por segurança)
train_end = "2022-12-31"
val_end = "2023-12-31"

train = df.loc[:train_end]
val = df.loc[train_end:val_end].iloc[1:]
test = df.loc[val_end:].iloc[1:]

X_train, y_train = create_windows(train["close"], LOOKBACK, HORIZON)
X_val, y_val     = create_windows(val["close"], LOOKBACK, HORIZON)
X_test, y_test   = create_windows(test["close"], LOOKBACK, HORIZON)

print("Shapes:")
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:  ", X_val.shape,   "y_val:  ", y_val.shape)
print("X_test: ", X_test.shape,  "y_test: ", y_test.shape)

Shapes:
X_train: (1229, 30) y_train: (1229,)
X_val:   (219, 30) y_val:   (219,)
X_test:  (463, 30) y_test:  (463,)


In [5]:
sample_idx = 0

print("Janela de entrada (X):")
print(X_train[sample_idx])

print("\nValor real do dia seguinte (y):")
print(y_train[sample_idx])

Janela de entrada (X):
[4.92853212 5.25289869 5.28058958 5.32533789 5.48850918 5.48702717
 5.53004456 5.53993177 5.51273632 5.44178247 5.55575657 5.54883337
 5.689013   5.77752113 5.90657616 5.82968807 5.84328508 6.0158515
 6.10287523 6.00077057 6.07691813 5.9458847  5.77331924 5.28330851
 5.57701778 5.65662432 5.37775087 5.73771715 5.63758898 5.75131559]

Valor real do dia seguinte (y):
5.968630313873291


In [6]:
def naive_forecast(X):
    """
    Usa o último valor da janela como previsão.
    """
    return X[:, -1]

In [7]:
def moving_average_forecast(X):
    """
    Usa a média da janela como previsão.
    """
    return X.mean(axis=1)

In [8]:
from sklearn.metrics import mean_absolute_error

y_pred_naive = naive_forecast(X_test)
y_pred_ma    = moving_average_forecast(X_test)

mae_naive = mean_absolute_error(y_test, y_pred_naive)
mae_ma    = mean_absolute_error(y_test, y_pred_ma)

print(f"Baseline Naive MAE: {mae_naive:.4f}")
print(f"Baseline Média Móvel MAE: {mae_ma:.4f}")

Baseline Naive MAE: 3.0149
Baseline Média Móvel MAE: 9.3359
